# AceIQ: floor and forecast upgrade
Version 2. Run from this repo in VS Code/Jupyter or a Databricks Git folder. This complements 01_data_checks and preserves its original results. All data is synthetic; no July actuals exist.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'dashboard').is_dir(), 'Set ROOT to the cloned project folder.'
sys.path.insert(0, str(ROOT))
from dashboard.forecast import build_forecasts, aggregate_predictions
from dashboard.analytics import machine_summary, floor_snapshot
from dashboard.decisions import DecisionEngine, rank_portfolios
data = ROOT / 'data'
daily = pd.read_parquet(data / 'machine_daily.parquet')
daily.gaming_date = pd.to_datetime(daily.gaming_date)
variance = pd.read_parquet(data / 'daily_variance.parquet')
variance.gaming_date = pd.to_datetime(variance.gaming_date)
daily = daily.merge(variance, validate='one_to_one')
machines = pd.read_parquet(data / 'machines.parquet')
prop = pd.read_parquet(data / 'property_daily.parquet')
report = json.loads((data / 'analysis_results.json').read_text())

## Forecast without test-outcome leakage
Fit January-March, select on April, refit January-April and freeze May-June predictions. Target: daily wagers per active machine. April empirical error bands are illustrative, not guaranteed confidence intervals.

In [ ]:
predictions, metadata = build_forecasts(daily, machines)
print(json.dumps(metadata, indent=2))
comparison = aggregate_predictions(predictions[predictions.phase == 'May–June holdout'], predictions[predictions.phase == 'April calibration'])
display(comparison)

## Time-valid floor
06:00 Las Vegas snapshot with end-exclusive placements and lifecycle eligibility.

In [ ]:
locations = pd.read_parquet(data / 'floor_locations.parquet')
placements = pd.read_parquet(data / 'placements.parquet')
floor = floor_snapshot(placements, locations, machines, '2025-07-01')
print('Occupied:', int(floor.machine_id.notna().sum()), 'Empty:', int(floor.machine_id.isna().sum()))
display(floor)

## Shared intervention scenarios
The same engine powers both map and portfolio UI. Catalog prices, lead times, manufacturer scope and eligibility are enforced. Intervention effects are assumptions; unpriced learning value is excluded.

In [ ]:
engine = DecisionEngine(daily, machines, prop, report, predictions)
candidates = engine.shortlist(report['simulation']['candidates'])
plans = rank_portfolios(candidates).sort_values('Expected net benefit', ascending=False)
display(plans.head(10))

## Explain the results
See docs/UPGRADE_GUIDE.md for validation, caveats and extension points. This notebook must be synced with its modules and data to Databricks explicitly; local edits do not change the cloud notebook.